# Prog10: Bidirectional Associative Memory (BAM)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Define pattern pairs for BAM


In [ ]:
# Define pattern pairs (X -> Y associations)
pattern_pairs = {
    'Pair1': (np.array([1, -1, 1, -1, 1]), np.array([-1, 1, -1])),
    'Pair2': (np.array([-1, 1, -1, 1, -1]), np.array([1, -1, 1])),
    'Pair3': (np.array([1, 1, -1, -1, 1]), np.array([1, 1, -1])),
    'Pair4': (np.array([-1, -1, 1, 1, -1]), np.array([-1, -1, 1]))
}

print("BAM Pattern Pairs:")
for name, (x_pattern, y_pattern) in pattern_pairs.items():
    print(f"{name}: {x_pattern} <-> {y_pattern}")


BAM Pattern Pairs:
Pair1: [ 1 -1  1 -1  1] <-> [-1  1 -1]
Pair2: [-1  1 -1  1 -1] <-> [ 1 -1  1]
Pair3: [ 1  1 -1 -1  1] <-> [ 1  1 -1]
Pair4: [-1 -1  1  1 -1] <-> [-1 -1  1]


# BAM Implementation


In [ ]:
class BAM:
    def __init__(self, x_size, y_size):
        self.x_size = x_size
        self.y_size = y_size
        self.weight_matrix = np.zeros((y_size, x_size))
        
    def store_patterns(self, x_patterns, y_patterns):
        """Store pattern pairs using Hebbian learning"""
        for x_pat, y_pat in zip(x_patterns, y_patterns):
            # Weight matrix: W = Y * X^T
            self.weight_matrix += np.outer(y_pat, x_pat)
    
    def forward_recall(self, x_input, max_iterations=10):
        """Recall Y from X input"""
        current_x = x_input.copy()
        current_y = np.zeros(self.y_size)
        
        for iteration in range(max_iterations):
            # X -> Y
            y_output = np.sign(np.dot(self.weight_matrix, current_x))
            
            # Y -> X
            x_output = np.sign(np.dot(self.weight_matrix.T, y_output))
            
            # Check for convergence
            if np.array_equal(x_output, current_x) and np.array_equal(y_output, current_y):
                break
                
            current_x = x_output
            current_y = y_output
            
        return current_x, current_y, iteration + 1
    
    def backward_recall(self, y_input, max_iterations=10):
        """Recall X from Y input"""
        current_y = y_input.copy()
        current_x = np.zeros(self.x_size)
        
        for iteration in range(max_iterations):
            # Y -> X
            x_output = np.sign(np.dot(self.weight_matrix.T, current_y))
            
            # X -> Y
            y_output = np.sign(np.dot(self.weight_matrix, x_output))
            
            # Check for convergence
            if np.array_equal(x_output, current_x) and np.array_equal(y_output, current_y):
                break
                
            current_x = x_output
            current_y = y_output
            
        return current_x, current_y, iteration + 1
    
    def get_energy(self, x_pattern, y_pattern):
        """Calculate energy of the pattern pair"""
        return -np.dot(y_pattern, np.dot(self.weight_matrix, x_pattern))


# Train BAM


In [ ]:
# Create and train BAM
x_patterns = [pair[0] for pair in pattern_pairs.values()]
y_patterns = [pair[1] for pair in pattern_pairs.values()]

bam = BAM(x_size=5, y_size=3)
bam.store_patterns(x_patterns, y_patterns)

print("BAM trained!")
print(f"Weight matrix shape: {bam.weight_matrix.shape}")
print(f"Weight matrix:\n{bam.weight_matrix}")


BAM trained!
Weight matrix shape: (3, 5)
Weight matrix:
[[ 0.  4. -4.  0.  0.]
 [ 4.  0.  0. -4.  4.]
 [-4.  0.  0.  4. -4.]]


# Test BAM Forward Recall (X -> Y)


In [ ]:
print("Testing BAM Forward Recall (X -> Y):")
for name, (original_x, original_y) in pattern_pairs.items():
    # Test with original X pattern
    recalled_x, recalled_y, iterations = bam.forward_recall(original_x)
    
    x_correct = np.array_equal(recalled_x, original_x)
    y_correct = np.array_equal(recalled_y, original_y)
    
    print(f"\n{name}:")
    print(f"  Original X: {original_x}")
    print(f"  Original Y: {original_y}")
    print(f"  Recalled X: {recalled_x}")
    print(f"  Recalled Y: {recalled_y}")
    print(f"  X Correct:  {x_correct}")
    print(f"  Y Correct:  {y_correct}")
    print(f"  Iterations: {iterations}")
    print(f"  Energy:     {bam.get_energy(recalled_x, recalled_y):.2f}")
    
    # Test with noisy X pattern
    def add_noise(pattern, noise_level=0.2):
        noisy = pattern.copy()
        n_noise = int(len(pattern) * noise_level)
        if n_noise > 0:
            noise_indices = np.random.choice(len(pattern), n_noise, replace=False)
            noisy[noise_indices] *= -1
        return noisy
    
    noisy_x = add_noise(original_x, noise_level=0.2)
    noisy_recalled_x, noisy_recalled_y, noisy_iterations = bam.forward_recall(noisy_x)
    
    noisy_x_correct = np.array_equal(noisy_recalled_x, original_x)
    noisy_y_correct = np.array_equal(noisy_recalled_y, original_y)
    
    print(f"  Noisy X:    {noisy_x}")
    print(f"  Noisy Recall X: {noisy_recalled_x}")
    print(f"  Noisy Recall Y: {noisy_recalled_y}")
    print(f"  Noisy X Correct: {noisy_x_correct}")
    print(f"  Noisy Y Correct: {noisy_y_correct}")


Testing BAM Forward Recall (X -> Y):

Pair1:
  Original X: [ 1 -1  1 -1  1]
  Original Y: [-1  1 -1]
  Recalled X: [ 1. -1.  1. -1.  1.]
  Recalled Y: [-1.  1. -1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy X:    [ 1 -1  1  1  1]
  Noisy Recall X: [ 1. -1.  1. -1.  1.]
  Noisy Recall Y: [-1.  1. -1.]
  Noisy X Correct: True
  Noisy Y Correct: True

Pair2:
  Original X: [-1  1 -1  1 -1]
  Original Y: [ 1 -1  1]
  Recalled X: [-1.  1. -1.  1. -1.]
  Recalled Y: [ 1. -1.  1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy X:    [-1  1 -1 -1 -1]
  Noisy Recall X: [-1.  1. -1.  1. -1.]
  Noisy Recall Y: [ 1. -1.  1.]
  Noisy X Correct: True
  Noisy Y Correct: True

Pair3:
  Original X: [ 1  1 -1 -1  1]
  Original Y: [ 1  1 -1]
  Recalled X: [ 1.  1. -1. -1.  1.]
  Recalled Y: [ 1.  1. -1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy X:    [ 1  1 -1  1  1]
  Noisy Recall X: [ 1

# Test BAM Backward Recall (Y -> X)


In [ ]:
print("\nTesting BAM Backward Recall (Y -> X):")
for name, (original_x, original_y) in pattern_pairs.items():
    # Test with original Y pattern
    recalled_x, recalled_y, iterations = bam.backward_recall(original_y)
    
    x_correct = np.array_equal(recalled_x, original_x)
    y_correct = np.array_equal(recalled_y, original_y)
    
    print(f"\n{name}:")
    print(f"  Original X: {original_x}")
    print(f"  Original Y: {original_y}")
    print(f"  Recalled X: {recalled_x}")
    print(f"  Recalled Y: {recalled_y}")
    print(f"  X Correct:  {x_correct}")
    print(f"  Y Correct:  {y_correct}")
    print(f"  Iterations: {iterations}")
    print(f"  Energy:     {bam.get_energy(recalled_x, recalled_y):.2f}")
    
    # Test with noisy Y pattern
    noisy_y = add_noise(original_y, noise_level=0.2)
    noisy_recalled_x, noisy_recalled_y, noisy_iterations = bam.backward_recall(noisy_y)
    
    noisy_x_correct = np.array_equal(noisy_recalled_x, original_x)
    noisy_y_correct = np.array_equal(noisy_recalled_y, original_y)
    
    print(f"  Noisy Y:    {noisy_y}")
    print(f"  Noisy Recall X: {noisy_recalled_x}")
    print(f"  Noisy Recall Y: {noisy_recalled_y}")
    print(f"  Noisy X Correct: {noisy_x_correct}")
    print(f"  Noisy Y Correct: {noisy_y_correct}")



Testing BAM Backward Recall (Y -> X):

Pair1:
  Original X: [ 1 -1  1 -1  1]
  Original Y: [-1  1 -1]
  Recalled X: [ 1. -1.  1. -1.  1.]
  Recalled Y: [-1.  1. -1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy Y:    [-1  1 -1]
  Noisy Recall X: [ 1. -1.  1. -1.  1.]
  Noisy Recall Y: [-1.  1. -1.]
  Noisy X Correct: True
  Noisy Y Correct: True

Pair2:
  Original X: [-1  1 -1  1 -1]
  Original Y: [ 1 -1  1]
  Recalled X: [-1.  1. -1.  1. -1.]
  Recalled Y: [ 1. -1.  1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy Y:    [ 1 -1  1]
  Noisy Recall X: [-1.  1. -1.  1. -1.]
  Noisy Recall Y: [ 1. -1.  1.]
  Noisy X Correct: True
  Noisy Y Correct: True

Pair3:
  Original X: [ 1  1 -1 -1  1]
  Original Y: [ 1  1 -1]
  Recalled X: [ 1.  1. -1. -1.  1.]
  Recalled Y: [ 1.  1. -1.]
  X Correct:  True
  Y Correct:  True
  Iterations: 2
  Energy:     -32.00
  Noisy Y:    [ 1  1 -1]
  Noisy Recall X: [ 1.  1. -1. -1.  1

# BAM Energy Analysis


In [ ]:
# Analyze energy levels for all stored patterns
print("BAM Energy Analysis:")
print("Stored patterns and their energies:")
for name, (x_pat, y_pat) in pattern_pairs.items():
    energy = bam.get_energy(x_pat, y_pat)
    print(f"{name}: Energy = {energy:.2f}")

# Test energy convergence
print(f"\nEnergy convergence test:")
test_x = np.array([1, -1, 1, -1, 1])  # One of our stored patterns
print(f"Initial X: {test_x}")

for i in range(5):
    recalled_x, recalled_y, iterations = bam.forward_recall(test_x)
    energy = bam.get_energy(recalled_x, recalled_y)
    print(f"Iteration {i+1}: X={recalled_x}, Y={recalled_y}, Energy={energy:.2f}")
    test_x = recalled_x


BAM Energy Analysis:
Stored patterns and their energies:
Pair1: Energy = -32.00
Pair2: Energy = -32.00
Pair3: Energy = -32.00
Pair4: Energy = -32.00

Energy convergence test:
Initial X: [ 1 -1  1 -1  1]
Iteration 1: X=[ 1. -1.  1. -1.  1.], Y=[-1.  1. -1.], Energy=-32.00
Iteration 2: X=[ 1. -1.  1. -1.  1.], Y=[-1.  1. -1.], Energy=-32.00
Iteration 3: X=[ 1. -1.  1. -1.  1.], Y=[-1.  1. -1.], Energy=-32.00
Iteration 4: X=[ 1. -1.  1. -1.  1.], Y=[-1.  1. -1.], Energy=-32.00
Iteration 5: X=[ 1. -1.  1. -1.  1.], Y=[-1.  1. -1.], Energy=-32.00
